In [1]:
### -----------------------------------------
### Step 2: Setup & Imports
### -----------------------------------------

# --- Data Handling ---
import pandas as pd
import numpy as np

# --- NLP (TF-IDF & Similarity) ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Computer Vision (PyTorch) ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# --- Image Handling ---
from PIL import Image

# --- OS Utilities ---
import os

# --- Utility / Helpers ---
from collections import Counter # Useful for finding top categories
from sklearn.model_selection import train_test_split # For splitting data

# Import the progress bar library
from tqdm import tqdm

#Second NLP model import
from sentence_transformers import SentenceTransformer

print("All libraries imported successfully.")

/Users/franekapanowicz/Python/ML_2025/machine-learning-project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully.


In [2]:
### -----------------------------------------
### Step 3: Load & Inspect the Dataset
### -----------------------------------------

# Define the path to your dataset
data_file = 'project_data_clean.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(data_file)

print(f"Successfully loaded '{data_file}'.")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("-" * 30)

# --- 1. Inspect Column Names ---
print("\n[INFO] All columns in the dataset:")
print(list(df.columns))
print("-" * 30)

# --- 2. Inspect Sample Data (the first 5 rows) ---
print("\n[INFO] Sample data (top 5 rows):")
print(df.head())
    

Successfully loaded 'project_data_clean.parquet'.
Dataset shape: 76935 rows, 11 columns
------------------------------

[INFO] All columns in the dataset:
['code', 'nutriscore_grade', 'product_name', 'ingredients_text', 'kcal_100g', 'proteins_100g', 'fat_100g', 'carbs_100g', 'categories_en', 'allergens_en', 'image_url']
------------------------------

[INFO] Sample data (top 5 rows):
            code nutriscore_grade  \
0  4056489135968                b   
1  4311501048030          unknown   
2  0845681001577          unknown   
3  3250392934392                e   
4  4335619058750                e   

                                        product_name  \
0                                     Potato Waffles   
1                                          Vollmilch   
2                                  Dairy Free Butter   
3  Tablette fourrée chocolat au lait caramel à la...   
4                                  Muesli Bar Banana   

                                    ingredients_text 

In [3]:
### -----------------------------------------
### PRE-STEP 4: Check for Missing Data (NaNs)
### -----------------------------------------

print("Checking for missing (NaN) values in key columns...\n")

# Define the columns we are most interested in
key_columns = ['ingredients_text', 'categories_en', 'image_url']

# Calculate and print NaN counts for these columns
for col in key_columns:
    if col in df.columns:
        nan_count = df[col].isna().sum()
        print(f"  Column '{col}': {nan_count} missing values")
    else:
        print(f"  [Warning] Column '{col}' not found in DataFrame.")

print("\n---")

# Optional: Get a summary for all columns
print("Total missing values for ALL columns:")
print(df.isna().sum())

Checking for missing (NaN) values in key columns...

  Column 'ingredients_text': 0 missing values
  Column 'categories_en': 10303 missing values
  Column 'image_url': 0 missing values

---
Total missing values for ALL columns:
code                    0
nutriscore_grade        0
product_name            0
ingredients_text        0
kcal_100g               0
proteins_100g           0
fat_100g                0
carbs_100g              0
categories_en       10303
allergens_en            0
image_url               0
dtype: int64


In [4]:
### -----------------------------------------
### Step 4.1: Build TF-IDF Representation
### -----------------------------------------

# The column containing the text we want to search
ingredients_column = 'ingredients_text'

print(f"Starting TF-IDF vectorization on '{ingredients_column}'...")

# Handle any potential (NaN) values - good practice even if 0
df[ingredients_column] = df[ingredients_column].fillna("")

# Initialize the TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

# Fit and transform the ingredients text
tfidf_matrix = tfidf_vectorizer.fit_transform(df[ingredients_column])

print("TF-IDF vectorization complete.")
print(f"Shape of the TF-IDF matrix: {tfidf_matrix.shape}")

Starting TF-IDF vectorization on 'ingredients_text'...
TF-IDF vectorization complete.
Shape of the TF-IDF matrix: (76935, 5000)


In [5]:
### -----------------------------------------
### Step 4.2: Create Search Functionality
### -----------------------------------------

def search_ingredients(query, vectorizer, matrix, dataframe, top_n=5):
    """
    Searches for products based on an ingredient query using TF-IDF.
    """
    
    # 1. Transform the user's query using the *already fitted* vectorizer
    query_vec = vectorizer.transform([query])
    
    # 2. Calculate cosine similarity between the query and all products
    #    This gives a score of how similar each product is to the query
    cosine_sim = cosine_similarity(query_vec, matrix).flatten()
    
    # 3. Get the indices of the top_n most similar products
    #    'argsort' sorts from least to most similar, so '[-top_n:]' gets the top N
    #    '[::-1]' reverses them to be from most to least similar
    top_indices = cosine_sim.argsort()[-top_n:][::-1]
    
    # 4. Get the similarity scores for these top products
    top_scores = cosine_sim[top_indices]
    
    # 5. Return the top results as a DataFrame
    #    We find the original product info using the 'top_indices'
    results_df = dataframe.iloc[top_indices].copy()
    results_df['search_score'] = top_scores
    
    return results_df[['product_name', 'ingredients_text', 'search_score']]

print("Search function 'search_ingredients' created.")

Search function 'search_ingredients' created.


In [6]:
### -----------------------------------------
### Step 4.3: Test the Baseline
### -----------------------------------------

print("Testing the search engine...")

# --- Test Query ---
test_query = "chocolate and almonds"

print(f"\nSearching for: '{test_query}'\n")

# Run the search
search_results = search_ingredients(test_query, 
                                    vectorizer=tfidf_vectorizer, 
                                    matrix=tfidf_matrix, 
                                    dataframe=df, 
                                    top_n=5)

# --- Display Results ---
if search_results.empty:
    print("No results found.")
else:
    print("--- Top 5 Results ---")
    
    # Print each result clearly
    for i, row in search_results.iterrows():
        print(f"Result {i+1} (Score: {row['search_score']:.4f})")
        print(f"  Product: {row['product_name']}")
        print(f"  Ingredients: {row['ingredients_text'][:150]}...") # Show first 150 chars
        print("-" * 20)

Testing the search engine...

Searching for: 'chocolate and almonds'

--- Top 5 Results ---
Result 56561 (Score: 0.7584)
  Product: Unsweetened crunchy almond butter
  Ingredients: Almonds....
--------------------
Result 71209 (Score: 0.7584)
  Product: Almonds sliced
  Ingredients: ALMONDS...
--------------------
Result 51618 (Score: 0.7584)
  Product: almonds
  Ingredients: Almonds...
--------------------
Result 45890 (Score: 0.7584)
  Product: Slivered Almonds
  Ingredients: Almonds, blanched....
--------------------
Result 39867 (Score: 0.7584)
  Product: Flaked almonds
  Ingredients: FLAKED ALMONDS...
--------------------


In [7]:
### -----------------------------------------
### Step 5.1 (Revised): Filter Noise, Select Top 20 Clean Classes
### -----------------------------------------

category_column = 'categories_en'
print(f"Original dataset size: {len(df)} rows")

# --- 1. Drop rows with missing categories ---
df_cv = df.dropna(subset=[category_column]).copy()
print(f"After dropping NaNs: {len(df_cv)} rows")

# --- 2. Define and apply cleaning function ---
def clean_category(cat_string):
    parts = cat_string.split(', ')
    return parts[-1].strip()

df_cv['clean_category'] = df_cv[category_column].apply(clean_category)
print("\nCreated 'clean_category' column.")

# --- 3. NEW: Define and remove "noise" and "generic" categories ---
# We must remove these before counting the Top 20.
# 'undefined' is useless.
# 'groceries', 'snacks', 'beverages', 'confectioneries', 'frozen foods' are too broad
# and will overlap with specific classes (like 'potato crisps' or 'sodas').

labels_to_exclude = [
    'undefined', 
    'groceries', 
    'snacks', 
    'beverages', 
    'confectioneries', 
    'frozen foods'
]

print(f"\nExcluding {len(labels_to_exclude)} generic/noise labels...")
# Keep only rows where the 'clean_category' is NOT in our exclusion list
df_cv_filtered = df_cv[~df_cv['clean_category'].isin(labels_to_exclude)].copy()
print(f"Size after excluding noise: {len(df_cv_filtered)} rows")


# --- 4. Find Top 20 most common from the *filtered* list ---
# This list will be much more specific and useful
clean_category_counts = Counter(df_cv_filtered['clean_category'])
top_20_clean_categories = clean_category_counts.most_common(20)
top_20_labels = [category for category, count in top_20_clean_categories]

print("\n--- NEW Top 20 Most Common *Specific* Categories ---")
for i, (category, count) in enumerate(top_20_clean_categories):
    print(f"  {i+1:2}. {category} (Count: {count})")
print("-" * 30)

# --- 5. Filter dataset to only these Top 20 clean categories ---
# We use our 'df_cv_filtered' which already has the noise removed
df_cv_final = df_cv_filtered[df_cv_filtered['clean_category'].isin(top_20_labels)].copy()
print(f"Final dataset size (Top 20 specific classes): {len(df_cv_final)} rows")

# --- 6. Assign numeric labels ---
# We create mappings based on the final, clean list
class_to_idx = {label: i for i, label in enumerate(top_20_labels)}
idx_to_class = {i: label for i, label in enumerate(top_20_labels)}

# Map the clean category to its number
df_cv_final['label'] = df_cv_final['clean_category'].map(class_to_idx)

print("\nCreated numeric 'label' column.")
print(df_cv_final[['product_name', 'clean_category', 'label']].head())

# Note: We now use 'df_cv_final' for all following CV steps

Original dataset size: 76935 rows
After dropping NaNs: 66632 rows

Created 'clean_category' column.

Excluding 6 generic/noise labels...
Size after excluding noise: 62207 rows

--- NEW Top 20 Most Common *Specific* Categories ---
   1. sweetened beverages (Count: 929)
   2. biscuits (Count: 760)
   3. candies (Count: 542)
   4. cheeses (Count: 485)
   5. breads (Count: 424)
   6. yogurts (Count: 395)
   7. dark chocolates (Count: 325)
   8. crackers appetizers (Count: 324)
   9. potato crisps (Count: 303)
  10. breakfast cereals (Count: 256)
  11. protein powders (Count: 246)
  12. sodas (Count: 245)
  13. pastas (Count: 234)
  14. ice creams (Count: 232)
  15. sausages (Count: 219)
  16. cakes (Count: 216)
  17. virgin olive oils (Count: 215)
  18. peanut butters (Count: 214)
  19. protein bars (Count: 214)
  20. bonbons (Count: 209)
------------------------------
Final dataset size (Top 20 specific classes): 6987 rows

Created numeric 'label' column.
                              pro

In [8]:
### -----------------------------------------
### Step 5.2: Split the Dataset (Confirmed)
### -----------------------------------------

data_to_split = df_cv_final

# Use 'code' (which has the ID) not 'image_url' (which has the link)
image_identifier_column = 'code' 
label_column = 'label'

# X is now the Series of product codes
X = data_to_split[image_identifier_column]
y = data_to_split[label_column]

# Create the 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Ensures a balanced split
)

print("Dataset split complete.")
print(f"  Using '{image_identifier_column}' as the image identifier.")
print(f"  Training samples: {len(X_train)} (X_train, y_train)")
print(f"  Test samples:     {len(X_test)} (X_test, y_test)")

Dataset split complete.
  Using 'code' as the image identifier.
  Training samples: 5589 (X_train, y_train)
  Test samples:     1398 (X_test, y_test)


In [9]:
### -----------------------------------------
### Step 5.3: Create a Custom Image Dataset Class (FIXED)
### -----------------------------------------

# --- 1. Define Image Transformations (No Change) ---
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# --- 2. Define the Custom Dataset Class (With Fix) ---
IMAGE_DIR = '../images/' 

class CustomImageDataset(Dataset):
    def __init__(self, image_paths, labels, image_dir, transform=None):
        self.image_paths = image_paths 
        self.labels = labels
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_code = self.image_paths.iloc[idx]
        
        # --- THIS IS THE FIX ---
        # This line was missing before, causing the 'label' not defined error
        label = self.labels.iloc[idx]
        # ---------------------
        
        img_filename = str(img_code) + '.jpg'
        img_path = os.path.join(self.image_dir, img_filename)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            # print(f"Warning: Image not found at {img_path}.") # You can un-comment this for debugging
            return torch.zeros(3, 224, 224), -1 
        except Exception as e:
            # print(f"Warning: Error loading {img_path}. Error: {e}.")
            return torch.zeros(3, 224, 224), -1

        if self.transform:
            image = self.transform(image)
            
        return image, label # This line will now work

print("CustomImageDataset class definition updated.")

# --- 3. Instantiate Train and Test Datasets ---
# We MUST re-create these objects to use the new class definition
train_dataset = CustomImageDataset(
    image_paths=X_train, 
    labels=y_train, 
    image_dir=IMAGE_DIR, 
    transform=data_transforms
)
test_dataset = CustomImageDataset(
    image_paths=X_test, 
    labels=y_test, 
    image_dir=IMAGE_DIR, 
    transform=data_transforms
)

print(f"\nSuccessfully re-created datasets:")
print(f"  Training dataset size: {len(train_dataset)}")
print(f"  Test dataset size:     {len(test_dataset)}")

CustomImageDataset class definition updated.

Successfully re-created datasets:
  Training dataset size: 5589
  Test dataset size:     1398


In [10]:
### -----------------------------------------
### Step 5.4: Define DataLoaders (FIXED)
### -----------------------------------------

# --- 1. Define 'collate_fn_safe' (No Change) ---
def collate_fn_safe(batch):
    """
    A custom collate function that filters out items that 
    failed to load in the Dataset class (which we flagged with label = -1).
    """
    batch_filtered = [(img, lbl) for img, lbl in batch if lbl != -1 and img.shape[0] == 3]
    
    if len(batch_filtered) == 0:
        return torch.empty(0, 3, 224, 224), torch.empty(0, dtype=torch.long)

    return torch.utils.data.dataloader.default_collate(batch_filtered)

print("Safe collate function 'collate_fn_safe' defined.")

# --- 2. Define the DataLoaders (With Fix) ---

BATCH_SIZE = 32

# --- THIS IS THE FIX ---
# Set num_workers=0 to avoid the multiprocessing error
NUM_WORKERS = 0 
# ---------------------

print(f"Using num_workers = {NUM_WORKERS} to avoid multiprocessing issues.")

# Create the Training DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,      
    num_workers=NUM_WORKERS, # Set to 0
    collate_fn=collate_fn_safe,
    pin_memory=True    
)

# Create the Test DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,     
    num_workers=NUM_WORKERS, # Set to 0
    collate_fn=collate_fn_safe,
    pin_memory=True
)

print(f"\nDataLoaders created with a batch size of {BATCH_SIZE}.")
print(f"  Train loader: {len(train_loader)} batches")
print(f"  Test loader:  {len(test_loader)} batches")

# --- 3. (Optional) Test the loader ---
try:
    print("\nTesting one batch from train_loader...")
    # Get one batch of data
    images, labels = next(iter(train_loader))
    
    print(f"  Successfully loaded one batch.")
    print(f"  Images tensor shape: {images.shape}")
    print(f"  Labels tensor shape: {labels.shape}")
except Exception as e:
    print(f"\n[Error] Could not load a batch: {e}")
    print("Please check your image paths and CustomImageDataset class.")

Safe collate function 'collate_fn_safe' defined.
Using num_workers = 0 to avoid multiprocessing issues.

DataLoaders created with a batch size of 32.
  Train loader: 175 batches
  Test loader:  44 batches

Testing one batch from train_loader...
  Successfully loaded one batch.
  Images tensor shape: torch.Size([32, 3, 224, 224])
  Labels tensor shape: torch.Size([32])


/Users/franekapanowicz/Python/ML_2025/machine-learning-project/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [11]:
### -----------------------------------------
### Step 5.5: Define a Simple CNN Architecture
### -----------------------------------------

# First, determine the number of classes we are predicting
# This is the len() of our top_20_labels list from Step 5.1
num_classes = len(class_to_idx)
print(f"Defining a CNN for {num_classes} classes.")

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        
        # --- Block 1 ---
        # Input: [3, 224, 224] (3 color channels)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Output: [16, 112, 112]

        # --- Block 2 ---
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Output: [32, 56, 56]

        # --- Block 3 ---
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Output: [64, 28, 28]

        # --- Flatten Layer ---
        # This "flattens" the 3D feature map into a 1D vector
        self.flatten = nn.Flatten()
        # Input: [64, 28, 28] -> Output: [64 * 28 * 28] = 50176

        # --- Fully Connected Layers ---
        self.fc1 = nn.Linear(in_features=64 * 28 * 28, out_features=512)
        self.relu_fc = nn.ReLU()
        
        # --- Output Layer ---
        # This is the final classification layer
        self.fc2 = nn.Linear(in_features=512, out_features=num_classes)

    def forward(self, x):
        # Pass input through the blocks
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        
        # Flatten the output
        x = self.flatten(x)
        
        # Pass through the classifier
        x = self.relu_fc(self.fc1(x))
        x = self.fc2(x) # Output is raw scores (logits)
        
        return x

# --- Instantiate the model ---
model = SimpleCNN(num_classes=num_classes)

# --- (Optional) Set device (GPU or CPU) ---
# This is a crucial step for training
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("\nUsing GPU: cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps") # For Apple Silicon (M1/M2/M3)
    print("\nUsing GPU: Apple MPS")
else:
    device = torch.device("cpu")
    print("\nUsing CPU")

# Move the model to the chosen device
model.to(device)

# (Optional) Print model summary to verify
print("\nModel architecture:")
print(model)

Defining a CNN for 20 classes.

Using GPU: Apple MPS

Model architecture:
SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu3): ReLU()
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=50176, out_features=512, bias=True)
  (relu_fc): ReLU()
  (fc2): Linear(in_features=512, out_features=20, bias=True)
)


In [12]:
### -----------------------------------------
### Step 5.6: Train the Baseline Model
### -----------------------------------------

# --- 1. Define Loss Function and Optimizer ---

# CrossEntropyLoss is the standard for multi-class classification.
# It combines LogSoftmax and NLLLoss in one step.
criterion = nn.CrossEntropyLoss()

# Adam is a popular, effective optimizer that works well with default settings.
# We pass it the model's parameters and a learning rate (lr).
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- 2. Define Training Parameters ---
num_epochs = 3 # As per the plan: train for 2-3 epochs
print(f"Starting training for {num_epochs} epochs...")

# --- 3. The Training Loop ---

# Set the model to "training mode" (this enables features like dropout, if we had them)
model.train()

for epoch in range(num_epochs):
    
    # We'll track the loss for this epoch
    running_loss = 0.0
    batches_processed = 0
    
    # Loop over the train_loader, which provides data in batches
    # tqdm wraps the loader to give us a nice progress bar
    for i, data in enumerate(tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}")):
        
        # 'data' is a list [images, labels]
        images, labels = data
        
        # Move the data to the same device as the model (e.g., GPU or CPU)
        images = images.to(device)
        labels = labels.to(device)

        # --- Forward Pass ---
        # 1. Zero the gradients: Clear old gradients from the last step
        optimizer.zero_grad()
        
        # 2. Pass images through the model
        outputs = model(images)
        
        # 3. Calculate the loss: How wrong was the model?
        loss = criterion(outputs, labels)
        
        # --- Backward Pass ---
        # 4. Calculate gradients: Figure out how much each weight contributed to the error
        loss.backward()
        
        # 5. Update weights: Adjust the model's weights to reduce the error
        optimizer.step()

        # --- Statistics ---
        running_loss += loss.item() # .item() gets the loss as a Python number
        batches_processed += 1

    # Print the average loss for the epoch
    avg_loss = running_loss / batches_processed
    print(f"  Epoch [{epoch + 1}/{num_epochs}] - Average Training Loss: {avg_loss:.4f}")

print("\nBaseline training complete.")

Starting training for 3 epochs...


Epoch 1/3: 100%|██████████| 175/175 [00:27<00:00,  6.29it/s]


  Epoch [1/3] - Average Training Loss: 2.9034


Epoch 2/3: 100%|██████████| 175/175 [00:24<00:00,  7.08it/s]


  Epoch [2/3] - Average Training Loss: 2.7609


Epoch 3/3: 100%|██████████| 175/175 [00:25<00:00,  6.84it/s]

  Epoch [3/3] - Average Training Loss: 2.5777

Baseline training complete.


In [13]:
### -----------------------------------------
### Step 5.7: Evaluate Performance
### -----------------------------------------

print("Starting evaluation on the test set...")

# Set the model to "evaluation mode"
# This turns off things like dropout and batch normalization
model.eval()

correct = 0
total = 0

# We don't need to track gradients during evaluation
with torch.no_grad():
    
    # Loop over the test_loader (no tqdm progress bar needed, it's fast)
    for data in test_loader:
        images, labels = data
        
        # Move data to the same device as the model
        images = images.to(device)
        labels = labels.to(device)
        
        # --- 1. Forward Pass ---
        # Get the model's raw score (logit) predictions
        outputs = model(images)
        
        # --- 2. Get Predictions ---
        # torch.max(outputs.data, 1) returns two things:
        # [0] the highest score (we ignore this)
        # [1] the index (the class number) of that highest score
        _, predicted = torch.max(outputs.data, 1)
        
        # --- 3. Tally Correct Guesses ---
        # Add the batch size to the total number of images seen
        total += labels.size(0)
        
        # Add the number of correct predictions in this batch
        correct += (predicted == labels).sum().item()

# --- 4. Calculate Final Accuracy ---
accuracy = 100 * correct / total
print("\nEvaluation complete.")
print(f"  Total correct: {correct}")
print(f"  Total images:  {total}")
print(f"  Accuracy on the test set: {accuracy:.2f} %")

Starting evaluation on the test set...

Evaluation complete.
  Total correct: 267
  Total images:  1397
  Accuracy on the test set: 19.11 %


In [14]:
### -----------------------------------------
### Step 6: Save Baseline Outputs (Path Fixed)
### -----------------------------------------
import pickle
import json

# --- 1. Create a directory for our saved models ---
# This path goes UP one level (..) from the 'notebooks' folder
# to the project root, then creates 'models'
output_dir = "../models"
os.makedirs(output_dir, exist_ok=True)
print(f"Created/verified output directory: {output_dir}")

# --- 2. Save the TF-IDF Vectorizer ---
tfidf_path = os.path.join(output_dir, "tfidf_vectorizer.pkl")
with open(tfidf_path, 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print(f"  [OK] TF-IDF vectorizer saved to: {tfidf_path}")

# --- 3. Save the trained CNN weights ---
cnn_weights_path = os.path.join(output_dir, "simple_cnn_weights.pth")
torch.save(model.state_dict(), cnn_weights_path)
print(f"  [OK] CNN model weights saved to: {cnn_weights_path}")

# --- 4. Save the Class Label Mappings ---
class_map_path = os.path.join(output_dir, "class_map.json")
with open(class_map_path, 'w') as f:
    json.dump(idx_to_class, f, indent=2)
print(f"  [OK] Class label mappings saved to: {class_map_path}")

print("\nAll baseline outputs have been saved.")

Created/verified output directory: ../models
  [OK] TF-IDF vectorizer saved to: ../models/tfidf_vectorizer.pkl
  [OK] CNN model weights saved to: ../models/simple_cnn_weights.pth
  [OK] Class label mappings saved to: ../models/class_map.json

All baseline outputs have been saved.


## Notebook 02: Baseline Model Development & Summary

This notebook establishes the baseline performance for our two core tasks: an **NLP search engine** for ingredients and a **Computer Vision (CV) model** for image classification.

The primary goal was not to achieve high accuracy, but to build a complete, end-to-end pipeline that successfully loads, processes, trains, and evaluates on our data.

### 1. Baseline NLP: Ingredient Search Engine

We built a search engine to find products based on their ingredients.

* **TF-IDF Representation:** A `TfidfVectorizer` was trained on the `ingredients_text` column, learning a vocabulary of the top 5,000 words.
* **Search Functionality:** A function using `cosine_similarity` was created to compare a user's query vector against the TF-IDF matrix of all products.
* **Validation:** The search was successfully tested with the query "chocolate and almonds" and returned relevant results.

### 2. Baseline CV: Image Classification (Top 20 Classes)

A complete pipeline was built to classify product images.

* **Data Cleaning & Preparation:** This was a critical step.
    * The hierarchical `categories_en` column was processed to extract a single, specific `clean_category` label for each product.
    * Generic and "noise" labels (e.g., `undefined`, `snacks`, `groceries`) were filtered out.
    * The **Top 20 most common, specific classes** (e.g., "biscuits," "cheeses," "sodas") were selected for our baseline task.
* **PyTorch Pipeline:**
    * A `train_test_split` (80/20) was performed with stratification to ensure balanced classes.
    * A custom `CustomImageDataset` class was built to load images from the `../images/` directory, correctly mapping the product `code` to the image filename (e.g., `<code>.jpg`).
    * `DataLoaders` were created with a `batch_size=32` and a custom `collate_fn_safe` to handle any missing/corrupt images.
* **Model & Training:**
    * A **Simple 3-Block CNN** was defined, instantiated, and moved to the `mps` (Apple GPU) device.
    * The model was trained for **3 epochs** using `CrossEntropyLoss` and the `Adam` optimizer.
    * Training was successful, with the loss decreasing from **2.9** to **2.57**.
* **Evaluation:**
    * The model achieved a baseline accuracy of **19.11%** on the test set. This is significantly better than random chance (5%) and confirms the pipeline is functional.

### 3. Key Outputs & Saved Artifacts

To make these models reusable, all key components were saved to the `models/` directory in the project root:

* **`models/tfidf_vectorizer.pkl`**: The saved TF-IDF vectorizer for the search engine.
* **`models/simple_cnn_weights.pth`**: The trained weights of our baseline CNN.
* **`models/class_map.json`**: A JSON file mapping the model's numerical outputs (e.g., `1`) back to human-readable labels (e.g., `"biscuits"`).

In [15]:
# Load a pre-trained model. 
# 'all-MiniLM-L6-v2' is a very popular, fast, and high-quality model.
print("Loading Sentence-BERT model 'all-MiniLM-L6-v2'...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Sentence-BERT model loaded successfully.")

Loading Sentence-BERT model 'all-MiniLM-L6-v2'...
Sentence-BERT model loaded successfully.
